In [2]:
!pip install gensim

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import BertTokenizer
import torch as pt
import re
import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
from tqdm import tqdm
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import copy
import random
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, average_precision_score

from gensim.models import Word2Vec
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# 1. Load and final procesisng of data
lemmatizer = WordNetLemmatizer()

#Lower case all text and lemmanise
def preprocess_text(text: str):
    text = text.lower()
    tokens = re.findall(r"[a-z]+", text)
    tokens = [lemmatizer.lemmatize(tok) for tok in tokens]
    return tokens

def load_texts_for_word2vec(csv_path):
    """Load texts from CSV and tokenize (lowercase + lemmatize) for Word2Vec training"""
    print(f"Loading data from {csv_path}...")
    df = pd.read_csv(csv_path)
    df['combined_text'] = df['combined_text'].astype(str)

    # Tokenize with preprocessing
    tokenized_texts = [preprocess_text(text) for text in df['combined_text']]
    labels = df['label'].values

    return tokenized_texts, labels, df

# 2. Train Word2Vec model
def train_word2vec(tokenized_texts, vector_size=100, window=5, min_count=2, workers=4, epochs=10):
    """
    Train Word2Vec model

    Parameters:
    - vector_size: dimensionality of word vectors (100, 200, 300)
    - window: context window size
    - min_count: ignores words with frequency lower than this
    - workers: number of worker threads
    - epochs: number of training epochs
    """
    print("Training Word2Vec model...")
    model = Word2Vec(
        sentences=tokenized_texts,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers,
        epochs=epochs,
        sg=1  # 1 for skip-gram, 0 for CBOW
    )
    print(f"Word2Vec model trained with vocabulary size: {len(model.wv)}")
    return model

# 3. Generate averaged Word2Vec features IN CHUNKS
def get_word2vec_features_chunked(tokenized_texts, w2v_model, labels, output_path, vector_size=100, chunk_size=1000):
    """Convert tokenized texts to averaged Word2Vec vectors and save in chunks"""

    # Create column names
    w2v_feature_cols = [f'w2v_{i}' for i in range(vector_size)]

    # Write header first
    header_df = pd.DataFrame(columns=w2v_feature_cols + ['label'])
    header_df.to_csv(output_path, index=False)

    # Process in chunks
    total_texts = len(tokenized_texts)
    num_chunks = (total_texts + chunk_size - 1) // chunk_size

    for chunk_idx in tqdm(range(num_chunks), desc="Processing chunks"):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, total_texts)

        # Process current chunk
        chunk_features = []
        for tokens in tokenized_texts[start_idx:end_idx]:
            # Get vectors for words in vocabulary
            valid_vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]

            if valid_vectors:
                # Average the vectors
                avg_vector = np.mean(valid_vectors, axis=0)
            else:
                # Zero vector if no words found
                avg_vector = np.zeros(vector_size)

            chunk_features.append(avg_vector)

        # Create DataFrame for this chunk
        chunk_array = np.array(chunk_features)
        chunk_df = pd.DataFrame(chunk_array, columns=w2v_feature_cols)
        chunk_df['label'] = labels[start_idx:end_idx]

        # Append to CSV (without header)
        chunk_df.to_csv(output_path, mode='a', header=False, index=False)

        # Clear memory
        del chunk_features, chunk_array, chunk_df

    print(f"Word2Vec features saved to {output_path}")

# 4. Complete pipeline with chunked saving
def create_word2vec_features(csv_path, output_path, vector_size=100, window=5, chunk_size=1000):
    """Complete pipeline to create Word2Vec features with memory-efficient saving"""

    # Load and tokenize
    tokenized_texts, labels, original_df = load_texts_for_word2vec(csv_path)

    # Train Word2Vec
    w2v_model = train_word2vec(
        tokenized_texts,
        vector_size=vector_size,
        window=window
    )

    # Generate and save features in chunks
    get_word2vec_features_chunked(
        tokenized_texts,
        w2v_model,
        labels,
        output_path,
        vector_size=vector_size,
        chunk_size=chunk_size
    )

    # Save the model
    model_path = output_path.replace('.csv', '.model')
    w2v_model.save(model_path)
    print(f"Word2Vec model saved to {model_path}")

    # Return model and first few rows for verification
    verification_df = pd.read_csv(output_path, nrows=5)

    return verification_df, w2v_model

In [5]:
# w2v_df, w2v_model = create_word2vec_features(
#     csv_path='final_datasets/Combined_Train.csv',
#     output_path='preprocessed_datasets/word2vec_features.csv',
#     vector_size=100,  # 100, 200, or 300
#     window=5,
#     chunk_size=100
# )

# print("\nWord2Vec features generated!")
# print(f"Shape: {w2v_df.shape}")
# w2v_df.head()

In [6]:
def scale_data(dataframe, oversample=False):
  x = dataframe[dataframe.columns[:-1]].values
  y = dataframe[dataframe.columns[-1]].values

  scaler = StandardScaler()
  x = scaler.fit_transform(x)

  if oversample:
    ros = RandomOverSampler()
    x, y = ros.fit_resample(x, y)

  data = np.hstack((x, np.reshape(y, (-1, 1))))

  return data, x, y

In [7]:
def split_dataset(df):
    train = df.sample(frac=0.8, random_state=42)
    test = df.drop(train.index)
    print(f"Total rows in dataset: {len(df)}")
    print(f"Total rows in train set: {len(train)}")
    print(f"Total rows in test set: {len(test)}")

    print("\nClass distribution in full dataset:")
    print(df['label'].value_counts())

    print("\nClass distribution in train set:")
    print(train['label'].value_counts())

    print("\nClass distribution in test set:")
    print(test['label'].value_counts())

    train, xtrain, ytrain = scale_data(train)
    test, xtest, ytest = scale_data(test)

    print("\n KNN \n")

    return train, xtrain, ytrain, test, xtest, ytest

In [8]:
def train_ml_models(xtrain, ytrain, xtest, ytest):
    print("\n KNN \n")
    knn_model = KNeighborsClassifier(n_neighbors=5)
    knn_model.fit(xtrain, ytrain)

    ypred = knn_model.predict(xtest)
    print(classification_report(ytest, ypred))

    print("\n Gaussian NB \n")
    nb_model = GaussianNB()
    nb_model = nb_model.fit(xtrain, ytrain)

    ypred = nb_model.predict(xtest)

    print(classification_report(ytest, ypred))

    print("\n Logistica Regression \n")
    lgr_model = LogisticRegression()
    lgr_model = lgr_model.fit(xtrain, ytrain)

    ypred = lgr_model.predict(xtest)

    print(classification_report(ytest, ypred))

    print("\n Support Vector Classifier \n")
    svc_model = SVC()
    svc_model = svc_model.fit(xtrain, ytrain)

    ypred = svc_model.predict(xtest)
    print(classification_report(ytest, ypred))

    print("\n XGBoost \n")
    xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    xgb_model = xgb_model.fit(xtrain, ytrain)

    ypred = xgb_model.predict(xtest)
    print(classification_report(ytest, ypred))

    print("\n Random Forest Classifier \n")
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(xtrain, ytrain)

    ypred = rf_model.predict(xtest)
    print(classification_report(ytest, ypred))

In [9]:
class CNNModel(nn.Module):
    def __init__(self, embedding_dim, num_filters=100, filter_sizes=[3, 4, 5]):
        super(CNNModel, self).__init__()
        self.embedding_dim = embedding_dim

        # Reshape input to (batch_size, 1, embedding_dim) for conv1d
        self.conv_layers = nn.ModuleList([
            nn.Conv1d(in_channels=1, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

        # Fully connected layers
        self.fc1 = nn.Linear(num_filters * len(filter_sizes), 64)
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch_size, embedding_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, embedding_dim)

        # Apply convolutions
        conv_outputs = []
        for conv in self.conv_layers:
            conv_out = self.relu(conv(x))  # (batch_size, num_filters, length)
            pooled = torch.max(conv_out, dim=2)[0]  # Max pooling
            conv_outputs.append(pooled)

        # Concatenate all conv outputs
        x = torch.cat(conv_outputs, dim=1)  # (batch_size, num_filters * len(filter_sizes))

        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.sigmoid(self.fc2(x))

        return x

In [10]:
class LSTMModel(nn.Module):
    def __init__(self, embedding_dim, hidden_dim=128, num_layers=2):
        super(LSTMModel, self).__init__()
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim,
                           num_layers=num_layers, batch_first=True, dropout=0.5)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.fc2 = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch_size, embedding_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, embedding_dim) - add sequence dimension

        # LSTM forward pass
        lstm_out, (hidden, cell) = self.lstm(x)  # hidden: (num_layers, batch_size, hidden_dim)

        # Use last hidden state
        x = hidden[-1]  # (batch_size, hidden_dim)

        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.sigmoid(self.fc2(x))

        return x

In [11]:
class CNNLSTMModel(nn.Module):
    def __init__(self, embedding_dim, num_filters=64, filter_sizes=[3, 4, 5],
                 hidden_dim=128, num_layers=1):
        super(CNNLSTMModel, self).__init__()
        self.embedding_dim = embedding_dim

        # CNN layers
        self.conv_layers = nn.ModuleList([
            nn.Conv1d(in_channels=1, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])

        self.relu = nn.ReLU()

        # LSTM layers - input is concatenated conv outputs
        cnn_output_dim = num_filters * len(filter_sizes)
        self.lstm = nn.LSTM(input_size=cnn_output_dim, hidden_size=hidden_dim,
                           num_layers=num_layers, batch_first=True, dropout=0.5)

        self.dropout = nn.Dropout(0.5)

        # Fully connected layers
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch_size, embedding_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, embedding_dim)

        # Apply CNN
        conv_outputs = []
        for conv in self.conv_layers:
            conv_out = self.relu(conv(x))  # (batch_size, num_filters, length)
            pooled = torch.max(conv_out, dim=2)[0]  # Max pooling
            conv_outputs.append(pooled)

        x = torch.cat(conv_outputs, dim=1)  # (batch_size, cnn_output_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, cnn_output_dim) - add sequence dimension

        # Apply LSTM
        lstm_out, (hidden, cell) = self.lstm(x)
        x = hidden[-1]  # (batch_size, hidden_dim)

        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.sigmoid(self.fc2(x))

        return x

In [12]:
def train_deep_learning_model(model, train_loader, test_loader, epochs=10, learning_rate=0.001, device='cpu'):
    """
    Train a PyTorch model and evaluate on test set.

    Parameters:
    - model: PyTorch model
    - train_loader: Training DataLoader
    - test_loader: Test DataLoader
    - epochs: Number of training epochs
    - learning_rate: Learning rate
    - device: 'cpu' or 'cuda'
    """

    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print(f"Training on device: {device}")
    print(f"Model: {model.__class__.__name__}")
    print("=" * 50)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device).float()
            batch_y = batch_y.to(device).float().unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # Evaluate on test set
        model.eval()
        test_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x = batch_x.to(device).float()
                batch_y = batch_y.to(device).float().unsqueeze(1)

                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                test_loss += loss.item()

                # Calculate accuracy
                predicted = (outputs > 0.5).float()
                correct += (predicted == batch_y).sum().item()
                total += batch_y.size(0)

        avg_train_loss = train_loss / len(train_loader)
        avg_test_loss = test_loss / len(test_loader)
        accuracy = correct / total

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f} | Accuracy: {accuracy:.4f}")

    print("=" * 50)
    print(f"{model.__class__.__name__} training completed!")

    return model

# ==================== Wrapper Function ====================
def train_all_deep_learning_models(xtrain, ytrain, xtest, ytest, epochs=10, batch_size=32):
    """
    Train CNN, LSTM, and CNN-LSTM models.

    Parameters:
    - xtrain, ytrain: Training data and labels
    - xtest, ytest: Test data and labels
    - epochs: Number of training epochs
    - batch_size: Batch size for DataLoader
    """

    # Determine device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Convert to PyTorch tensors
    xtrain_tensor = torch.from_numpy(xtrain).float()
    ytrain_tensor = torch.from_numpy(ytrain).long()
    xtest_tensor = torch.from_numpy(xtest).float()
    ytest_tensor = torch.from_numpy(ytest).long()

    # Create DataLoaders
    train_dataset = TensorDataset(xtrain_tensor, ytrain_tensor)
    test_dataset = TensorDataset(xtest_tensor, ytest_tensor)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    embedding_dim = xtrain.shape[1]

    # Train CNN
    print("\n" + "=" * 50)
    print("TRAINING CNN MODEL")
    print("=" * 50)
    cnn_model = CNNModel(embedding_dim)
    cnn_model = train_deep_learning_model(cnn_model, train_loader, test_loader, epochs=epochs, device=device)

    # Train LSTM
    print("\n" + "=" * 50)
    print("TRAINING LSTM MODEL")
    print("=" * 50)
    lstm_model = LSTMModel(embedding_dim)
    lstm_model = train_deep_learning_model(lstm_model, train_loader, test_loader, epochs=epochs, device=device)

    # Train CNN-LSTM
    print("\n" + "=" * 50)
    print("TRAINING CNN-LSTM HYBRID MODEL")
    print("=" * 50)
    cnn_lstm_model = CNNLSTMModel(embedding_dim)
    cnn_lstm_model = train_deep_learning_model(cnn_lstm_model, train_loader, test_loader, epochs=epochs, device=device)

    return cnn_model, lstm_model, cnn_lstm_model

In [13]:
def train_deep_learning_model(model, train_loader, val_loader, test_loader, epochs=10, learning_rate=0.001, device='cpu'):
    """
    Train a PyTorch model with validation set and evaluate on test set.

    Parameters:
    - model: PyTorch model
    - train_loader: Training DataLoader
    - val_loader: Validation DataLoader
    - test_loader: Test DataLoader (NOT USED during training, only for final evaluation)
    - epochs: Number of training epochs
    - learning_rate: Learning rate
    - device: 'cpu' or 'cuda'
    """

    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print(f"Training on device: {device}")
    print(f"Model: {model.__class__.__name__}")
    print("=" * 50)

    # Storage for history
    train_losses = []
    val_losses = []
    val_accuracies = []

    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device).float()
            batch_y = batch_y.to(device).float().unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # ==================== VALIDATION ====================
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device).float()
                batch_y = batch_y.to(device).float().unsqueeze(1)

                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()

                # Calculate accuracy
                predicted = (outputs > 0.5).float()
                val_correct += (predicted == batch_y).sum().item()
                val_total += batch_y.size(0)

        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = val_correct / val_total
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Accuracy: {val_accuracy:.4f}")

    print("=" * 50)
    print(f"{model.__class__.__name__} training completed!")

    # Test-set eval
    print("\n" + "=" * 50)
    print(f"FINAL EVALUATION ON TEST SET - {model.__class__.__name__}")
    print("=" * 50)

    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device).float()
            batch_y = batch_y.to(device).float().unsqueeze(1)

            outputs = model(batch_x)
            predicted = (outputs > 0.5).float()

            all_predictions.extend(predicted.cpu().numpy().flatten())
            all_labels.extend(batch_y.cpu().numpy().flatten())
            all_probabilities.extend(outputs.cpu().numpy().flatten())

    # Calculate metrics
    all_predictions = np.array(all_predictions)
    all_labels = np.array(all_labels)
    all_probabilities = np.array(all_probabilities)

    test_accuracy = np.mean(all_predictions == all_labels)
    test_auc = roc_auc_score(all_labels, all_probabilities)

    print(f"\nTest Accuracy: {test_accuracy:.4f}")
    print(f"Test AUC Score: {test_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_predictions, target_names=['Fake (0)', 'Real (1)']))

    # Visualisation
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Loss curves
    axes[0, 0].plot(train_losses, label='Training Loss', marker='o')
    axes[0, 0].plot(val_losses, label='Validation Loss', marker='s')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title(f'{model.__class__.__name__} - Loss over Epochs')
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # Accuracy curve
    axes[0, 1].plot(val_accuracies, label='Validation Accuracy', marker='o', color='green')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].set_title(f'{model.__class__.__name__} - Accuracy over Epochs')
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_predictions)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0], cbar=False)
    axes[1, 0].set_xlabel('Predicted')
    axes[1, 0].set_ylabel('Actual')
    axes[1, 0].set_title(f'{model.__class__.__name__} - Confusion Matrix (Test Set)')
    axes[1, 0].set_xticklabels(['Fake (0)', 'Real (1)'])
    axes[1, 0].set_yticklabels(['Fake (0)', 'Real (1)'])

    # ROC Curve
    fpr, tpr, _ = roc_curve(all_labels, all_probabilities)
    axes[1, 1].plot(fpr, tpr, label=f'ROC Curve (AUC = {test_auc:.4f})', color='darkorange', lw=2)
    axes[1, 1].plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
    axes[1, 1].set_xlabel('False Positive Rate')
    axes[1, 1].set_ylabel('True Positive Rate')
    axes[1, 1].set_title(f'{model.__class__.__name__} - ROC Curve (Test Set)')
    axes[1, 1].legend()
    axes[1, 1].grid(True)

    plt.tight_layout()
    plt.show()

    return model, {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_accuracies': val_accuracies,
        'test_accuracy': test_accuracy,
        'test_auc': test_auc,
        'predictions': all_predictions,
        'labels': all_labels,
        'probabilities': all_probabilities
    }

In [14]:
def train_all_deep_learning_models(xtrain, ytrain, xtest, ytest, epochs=10, batch_size=32, val_split=0.2):
    """
    Train CNN, LSTM, and CNN-LSTM models with proper train/validation/test split.

    Parameters:
    - xtrain, ytrain: Training data and labels
    - xtest, ytest: Test data and labels (NOT used during training)
    - epochs: Number of training epochs
    - batch_size: Batch size for DataLoader
    - val_split: Fraction of training data to use for validation (default: 0.2)

    Verification:
    - Train set is partitioned into training and validation
    - Test set is NOT used during training, only for final evaluation
    """

    print("=" * 70)
    print("DATA SPLIT VERIFICATION")
    print("=" * 70)
    print(f"Training set size: {xtrain.shape[0]}")
    print(f"Test set size: {xtest.shape[0]}")
    print(f"Validation split: {val_split * 100}%")
    print(f"Effective training size: {int(xtrain.shape[0] * (1 - val_split))}")
    print(f"Effective validation size: {int(xtrain.shape[0] * val_split)}")
    print("✓ Test set will NOT be used during training phase")
    print("✓ Test set will ONLY be used for final evaluation")
    print("=" * 70 + "\n")

    # Determine device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Split training data into train and validation
    from sklearn.model_selection import train_test_split as sklearn_train_test_split
    xtrain_split, xval, ytrain_split, yval = sklearn_train_test_split(
        xtrain, ytrain, test_size=val_split, random_state=42, stratify=ytrain
    )

    # Convert to PyTorch tensors
    xtrain_tensor = torch.from_numpy(xtrain_split).float()
    ytrain_tensor = torch.from_numpy(ytrain_split).long()
    xval_tensor = torch.from_numpy(xval).float()
    yval_tensor = torch.from_numpy(yval).long()
    xtest_tensor = torch.from_numpy(xtest).float()
    ytest_tensor = torch.from_numpy(ytest).long()

    # Create DataLoaders
    train_dataset = TensorDataset(xtrain_tensor, ytrain_tensor)
    val_dataset = TensorDataset(xval_tensor, yval_tensor)
    test_dataset = TensorDataset(xtest_tensor, ytest_tensor)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    embedding_dim = xtrain.shape[1]

    models_history = {}

    # Train CNN
    print("\n" + "=" * 70)
    print("TRAINING CNN MODEL")
    print("=" * 70)
    cnn_model = CNNModel(embedding_dim)
    cnn_model, cnn_history = train_deep_learning_model(
        cnn_model, train_loader, val_loader, test_loader,
        epochs=epochs, device=device
    )
    models_history['CNN'] = cnn_history

    # Train LSTM
    print("\n" + "=" * 70)
    print("TRAINING LSTM MODEL")
    print("=" * 70)
    lstm_model = LSTMModel(embedding_dim)
    lstm_model, lstm_history = train_deep_learning_model(
        lstm_model, train_loader, val_loader, test_loader,
        epochs=epochs, device=device
    )
    models_history['LSTM'] = lstm_history

    # Train CNN-LSTM
    print("\n" + "=" * 70)
    print("TRAINING CNN-LSTM HYBRID MODEL")
    print("=" * 70)
    cnn_lstm_model = CNNLSTMModel(embedding_dim)
    cnn_lstm_model, cnn_lstm_history = train_deep_learning_model(
        cnn_lstm_model, train_loader, val_loader, test_loader,
        epochs=epochs, device=device
    )
    models_history['CNN-LSTM'] = cnn_lstm_history

    return cnn_model, lstm_model, cnn_lstm_model, models_history

In [15]:
def run_complete_pipeline_word2vec(
    dataset_csv_path,
    vector_size=300,
    window=5,
    chunk_size=1000,
    regenerate_features=True,
    train_ml=True,
    train_dl=True,
    dl_epochs=10,
    dl_batch_size=32,
    val_split=0.2,
    save_vectors=0,  # 0 = do not save vectors CSV, 1 = save
    w2v_output_path='word2vec_features.csv'
):
    """
    End-to-end pipeline for Word2Vec features → ML models → Deep learning models.
    - If regenerate_features is True (or file missing), trains Word2Vec and saves averaged features (when save_vectors==1).
    - Otherwise, loads the existing features CSV (when save_vectors==1).
    - When save_vectors==0, features are built in-memory and not written to disk.
    """
    print("\n" + "=" * 80)
    print("Starting Word2Vec + ML + DL pipeline")
    print("=" * 80 + "\n")

    # Step 1: Build or load Word2Vec features
    if save_vectors:
        if regenerate_features or not os.path.exists(w2v_output_path):
            print("Step 1: Training Word2Vec and creating features (saving to CSV)")
            print("-" * 80)
            _preview_df, _w2v_model = create_word2vec_features(
                csv_path=dataset_csv_path,
                output_path=w2v_output_path,
                vector_size=vector_size,
                window=window,
                chunk_size=chunk_size
            )
            feature_df = pd.read_csv(w2v_output_path)
        else:
            print("Step 1: Loading existing Word2Vec features")
            print("-" * 80)
            feature_df = pd.read_csv(w2v_output_path)
            print(f"Loaded features from {w2v_output_path}")
    else:
        print("Step 1: Training Word2Vec and creating features (in-memory, not saving CSV)")
        print("-" * 80)
        tokenized_texts, labels, _ = load_texts_for_word2vec(dataset_csv_path)
        w2v_model = train_word2vec(
            tokenized_texts,
            vector_size=vector_size,
            window=window
        )
        w2v_feature_cols = [f"w2v_{i}" for i in range(vector_size)]
        features = []
        for tokens in tokenized_texts:
            valid_vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
            if valid_vectors:
                features.append(np.mean(valid_vectors, axis=0))
            else:
                features.append(np.zeros(vector_size))
        feature_df = pd.DataFrame(np.array(features), columns=w2v_feature_cols)
        feature_df['label'] = labels
        print("Features built in memory (not saved).")

    # Step 2: Split dataset
    print("\nStep 2: Splitting dataset (80/20 train/test)")
    print("-" * 80)
    train, xtrain, ytrain, test, xtest, ytest = split_dataset(feature_df)

    # Step 3: Train ML models
    if train_ml:
        print("\nStep 3: Training traditional ML models")
        print("-" * 80)
        train_ml_models(xtrain, ytrain, xtest, ytest)
    else:
        print("\nStep 3: Skipping traditional ML models (train_ml=False)")

    # Step 4: Train deep learning models
    models_history = {}
    if train_dl:
        print("\nStep 4: Training deep learning models with validation/test split")
        print("-" * 80)
        cnn_model, lstm_model, cnn_lstm_model, models_history = train_all_deep_learning_models(
            xtrain, ytrain, xtest, ytest,
            epochs=dl_epochs,
            batch_size=dl_batch_size,
            val_split=val_split
        )
    else:
        print("\nStep 4: Skipping deep learning models (train_dl=False)")

    # Summary
    print("\n" + "=" * 80)
    print("Pipeline completed successfully")
    print("=" * 80)
    print(f"Word2Vec feature size: {xtrain.shape[1]} dimensions")
    print(f"Training samples: {len(xtrain)}")
    print(f"Test samples: {len(xtest)}")
    if train_ml:
        print("ML models trained (KNN, Naive Bayes, Logistic Regression, SVC, XGBoost, Random Forest)")
    if train_dl:
        print("DL models trained (CNN, LSTM, CNN-LSTM)")
        print(f"  - CNN test accuracy: {models_history['CNN']['test_accuracy']:.4f}")
        print(f"  - LSTM test accuracy: {models_history['LSTM']['test_accuracy']:.4f}")
        print(f"  - CNN-LSTM test accuracy: {models_history['CNN-LSTM']['test_accuracy']:.4f}")
    print("=" * 80 + "\n")

    return feature_df, xtrain, ytrain, xtest, ytest, models_history

In [ ]:
# Averaged Word2Vec features for ML + sequence inputs for DL
def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def create_stratified_splits(df, label_col='label', test_size=0.2, val_size=0.2, seed=42):
    y = df[label_col].values
    all_indices = np.arange(len(df))

    train_idx, test_idx = train_test_split(
        all_indices,
        test_size=test_size,
        random_state=seed,
        stratify=y
    )

    y_train = y[train_idx]
    train_idx, val_idx = train_test_split(
        train_idx,
        test_size=val_size,
        random_state=seed,
        stratify=y_train
    )

    split_info = {
        'train_idx': train_idx,
        'val_idx': val_idx,
        'test_idx': test_idx
    }

    print('Split summary (stratified):')
    print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

    return split_info


def train_word2vec_model(tokenized_texts, vector_size=300, window=5, min_count=2, epochs=10, seed=42):
    model = Word2Vec(
        sentences=tokenized_texts,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=1,
        epochs=epochs,
        sg=1,
        seed=seed
    )
    return model


def average_word2vec_vector(tokens, w2v_model, vector_size):
    valid_vectors = [w2v_model.wv[tok] for tok in tokens if tok in w2v_model.wv]
    if not valid_vectors:
        return np.zeros(vector_size, dtype=np.float32)
    return np.mean(valid_vectors, axis=0).astype(np.float32)


def build_ml_matrices(tokenized_texts, labels, split_info, w2v_model, vector_size):
    features = np.array([
        average_word2vec_vector(tokens, w2v_model, vector_size)
        for tokens in tokenized_texts
    ], dtype=np.float32)

    y = np.array(labels)

    train_idx = split_info['train_idx']
    val_idx = split_info['val_idx']
    test_idx = split_info['test_idx']

    x_train = features[train_idx]
    y_train = y[train_idx]
    x_val = features[val_idx]
    y_val = y[val_idx]
    x_test = features[test_idx]
    y_test = y[test_idx]

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_val = scaler.transform(x_val)
    x_test = scaler.transform(x_test)

    return {
        'x_train': x_train,
        'y_train': y_train,
        'x_val': x_val,
        'y_val': y_val,
        'x_test': x_test,
        'y_test': y_test,
        'scaler': scaler
    }


def evaluate_classifier(model, x, y):
    y_pred = model.predict(x)

    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(x)[:, 1]
    elif hasattr(model, 'decision_function'):
        y_score = model.decision_function(x)
    else:
        y_score = y_pred

    precision, recall, f1, _ = precision_recall_fscore_support(y, y_pred, average='binary', zero_division=0)

    return {
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc_score(y, y_score),
        'pr_auc': average_precision_score(y, y_score)
    }


def train_ml_models_v2(ml_data, seed=42):
    x_train, y_train = ml_data['x_train'], ml_data['y_train']
    x_test, y_test = ml_data['x_test'], ml_data['y_test']

    models = {
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'GaussianNB': GaussianNB(),
        'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=seed),
        'SVC': SVC(probability=True, class_weight='balanced', random_state=seed),
        'XGBoost': XGBClassifier(
            eval_metric='logloss',
            random_state=seed,
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6
        ),
        'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=seed)
    }

    results = []
    fitted_models = {}

    for name, model in models.items():
        model.fit(x_train, y_train)
        fitted_models[name] = model
        metrics = evaluate_classifier(model, x_test, y_test)
        metrics['model'] = name
        results.append(metrics)

    result_df = pd.DataFrame(results).sort_values('f1', ascending=False)
    print('\nML results (test set):')
    print(result_df[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']])

    return fitted_models, result_df


def build_vocab(train_tokenized_texts, min_freq=2):
    counter = Counter(tok for tokens in train_tokenized_texts for tok in tokens)
    vocab = {'<PAD>': 0, '<UNK>': 1}

    for token, freq in counter.items():
        if freq >= min_freq:
            vocab[token] = len(vocab)

    return vocab


def encode_tokens(tokens, vocab, max_len):
    ids = [vocab.get(tok, vocab['<UNK>']) for tok in tokens]
    ids = ids[:max_len]
    if len(ids) < max_len:
        ids.extend([vocab['<PAD>']] * (max_len - len(ids)))
    return ids


def build_embedding_matrix(vocab, w2v_model, embedding_dim):
    matrix = np.random.normal(0, 0.1, (len(vocab), embedding_dim)).astype(np.float32)
    matrix[vocab['<PAD>']] = np.zeros(embedding_dim, dtype=np.float32)

    for token, idx in vocab.items():
        if token in ('<PAD>', '<UNK>'):
            continue
        if token in w2v_model.wv:
            matrix[idx] = w2v_model.wv[token]

    return matrix


def build_sequence_dataloaders(tokenized_texts, labels, split_info, vocab, max_len=200, batch_size=32):
    y = np.array(labels, dtype=np.float32)
    x = np.array([encode_tokens(tokens, vocab, max_len) for tokens in tokenized_texts], dtype=np.int64)

    train_idx = split_info['train_idx']
    val_idx = split_info['val_idx']
    test_idx = split_info['test_idx']

    x_train, y_train = x[train_idx], y[train_idx]
    x_val, y_val = x[val_idx], y[val_idx]
    x_test, y_test = x[test_idx], y[test_idx]

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
        batch_size=batch_size,
        shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_val), torch.from_numpy(y_val)),
        batch_size=batch_size,
        shuffle=False
    )
    test_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test)),
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader, val_loader, test_loader


class TextCNN(nn.Module):
    def __init__(self, embedding_matrix, num_filters=128, filter_sizes=(3, 4, 5), dropout=0.5):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=k)
            for k in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), 1)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        emb = emb.transpose(1, 2)

        conv_outputs = []
        for conv in self.convs:
            out = torch.relu(conv(emb))
            pooled = torch.max(out, dim=2)[0]
            conv_outputs.append(pooled)

        x = torch.cat(conv_outputs, dim=1)
        x = self.dropout(x)
        return self.fc(x).squeeze(1)


class TextLSTM(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_layers=2, dropout=0.5):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        lstm_out, _ = self.lstm(emb)
        x = lstm_out[:, -1, :]
        x = self.dropout(x)
        return self.fc(x).squeeze(1)


class TextCNNLSTM(nn.Module):
    def __init__(self, embedding_matrix, num_filters=128, kernel_size=3, hidden_dim=128, dropout=0.5):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

        self.conv = nn.Conv1d(embedding_dim, num_filters, kernel_size=kernel_size, padding=kernel_size // 2)
        self.lstm = nn.LSTM(
            input_size=num_filters,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        x = emb.transpose(1, 2)
        x = torch.relu(self.conv(x))
        x = x.transpose(1, 2)

        lstm_out, _ = self.lstm(x)
        x = self.dropout(lstm_out[:, -1, :])
        return self.fc(x).squeeze(1)


def _evaluate_dl(model, loader, criterion, device):
    model.eval()
    loss_sum = 0.0
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_x)
            loss = criterion(logits, batch_y)

            loss_sum += loss.item()
            all_logits.extend(logits.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    all_logits = np.array(all_logits)
    all_labels = np.array(all_labels)
    all_probs = 1.0 / (1.0 + np.exp(-all_logits))
    all_preds = (all_probs >= 0.5).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary', zero_division=0
    )

    metrics = {
        'loss': loss_sum / max(1, len(loader)),
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc_score(all_labels, all_probs),
        'pr_auc': average_precision_score(all_labels, all_probs),
        'labels': all_labels,
        'preds': all_preds,
        'probs': all_probs
    }

    return metrics


In [19]:
def train_dl_model_v2(model, train_loader, val_loader, test_loader, epochs=10, lr=1e-3, patience=3, device='cpu'):
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float('inf')
    best_state = None
    best_epoch = -1
    wait = 0

    history = {
        'train_loss': [],
        'val_loss': [],
        'val_f1': []
    }

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss = train_loss / max(1, len(train_loader))
        val_metrics = _evaluate_dl(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['val_f1'].append(val_metrics['f1'])

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val F1: {val_metrics['f1']:.4f}"
        )

        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping triggered at epoch {epoch + 1}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = _evaluate_dl(model, test_loader, criterion, device)
    test_metrics['best_epoch'] = best_epoch

    return model, history, test_metrics

def train_dl_models_v2(train_loader, val_loader, test_loader, embedding_matrix, epochs=10, lr=1e-3, patience=3):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    models = {
        'TextCNN': TextCNN(embedding_matrix),
        'TextLSTM': TextLSTM(embedding_matrix),
        'TextCNNLSTM': TextCNNLSTM(embedding_matrix)
    }

    results = []
    trained = {}

    for name, model in models.items():
        print(f"\nTraining {name} on sequence inputs...")
        trained_model, history, test_metrics = train_dl_model_v2(
            model,
            train_loader,
            val_loader,
            test_loader,
            epochs=epochs,
            lr=lr,
            patience=patience,
            device=device
        )
        trained[name] = {
            'model': trained_model,
            'history': history,
            'test_metrics': test_metrics
        }

        results.append({
            'model': name,
            'accuracy': test_metrics['accuracy'],
            'precision': test_metrics['precision'],
            'recall': test_metrics['recall'],
            'f1': test_metrics['f1'],
            'roc_auc': test_metrics['roc_auc'],
            'pr_auc': test_metrics['pr_auc'],
            'best_epoch': test_metrics['best_epoch']
        })

    results_df = pd.DataFrame(results).sort_values('f1', ascending=False)
    print('\nDL results (test set):')
    print(results_df)

    return trained, results_df

In [17]:
def run_hybrid_pipeline_v2(
    dataset_csv_path,
    vector_size=300,
    window=5,
    min_count=2,
    w2v_epochs=10,
    max_len=200,
    test_size=0.2,
    val_size=0.2,
    dl_epochs=10,
    dl_batch_size=32,
    dl_lr=1e-3,
    dl_patience=3,
    seed=42,
    transductive_w2v=False
):
    set_global_seed(seed)

    df = pd.read_csv(dataset_csv_path)
    df['combined_text'] = df['combined_text'].astype(str)

    tokenized_texts = [preprocess_text(text) for text in df['combined_text']]
    labels = df['label'].values

    split_info = create_stratified_splits(
        df,
        label_col='label',
        test_size=test_size,
        val_size=val_size,
        seed=seed
    )

    if transductive_w2v:
        w2v_train_tokens = tokenized_texts
        regime = 'transductive'
    else:
        w2v_train_tokens = [tokenized_texts[i] for i in split_info['train_idx']]
        regime = 'strict'

    print(f"\nWord2Vec regime: {regime}")
    w2v_model = train_word2vec_model(
        w2v_train_tokens,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        epochs=w2v_epochs,
        seed=seed
    )

    ml_data = build_ml_matrices(tokenized_texts, labels, split_info, w2v_model, vector_size)
    ml_models, ml_results = train_ml_models_v2(ml_data, seed=seed)

    train_tokens = [tokenized_texts[i] for i in split_info['train_idx']]
    vocab = build_vocab(train_tokens, min_freq=min_count)
    embedding_matrix = build_embedding_matrix(vocab, w2v_model, vector_size)

    train_loader, val_loader, test_loader = build_sequence_dataloaders(
        tokenized_texts,
        labels,
        split_info,
        vocab,
        max_len=max_len,
        batch_size=dl_batch_size
    )

    dl_models, dl_results = train_dl_models_v2(
        train_loader,
        val_loader,
        test_loader,
        embedding_matrix,
        epochs=dl_epochs,
        lr=dl_lr,
        patience=dl_patience
    )

    return {
        'split_info': split_info,
        'w2v_model': w2v_model,
        'word2vec_regime': regime,
        'ml_models': ml_models,
        'ml_results': ml_results,
        'dl_models': dl_models,
        'dl_results': dl_results,
        'vocab': vocab
    }

In [ ]:
result_bundle = run_hybrid_pipeline_v2(
    dataset_csv_path='/content/drive/MyDrive/datasets/WELFake_processed.csv',
    vector_size=300,
    window=5,
    min_count=2,
    w2v_epochs=10,
    max_len=200,
    test_size=0.2,
    val_size=0.2,
    dl_epochs=10,
    dl_batch_size=32,
    dl_lr=1e-3,
    dl_patience=3,
    seed=42,
    transductive_w2v=False
)

print('\nML Results Table:')
display(result_bundle['ml_results'])

print('\nDL Results Table:')
display(result_bundle['dl_results'])

print('\nWord2Vec Regime:', result_bundle['word2vec_regime'])
print('Vocab Size:', len(result_bundle['vocab']))

Split summary (stratified):
Train: 40748 | Val: 10188 | Test: 12734

Word2Vec regime: strict


: 